# Testing SeisBench: PhaseNet & EQTransformer

Smoke test for the `gpu-container` image: confirms `seisbench` imports, loads both **PhaseNet** and **EQTransformer** pretrained models, moves them to the GPU if one is available (exercising the `torch==2.13.0+cu126` pin against the image's baked-in driver), and runs real phase-picking inference end to end.

## Suggested test data

The data cell below fetches the same waveform SeisBench's own official example uses: about 1 hour of station **MN.AQU** (L'Aquila, Italy) starting at the 2009-04-06 *Mw 6.3* L'Aquila mainshock, pulled live from the **INGV** FDSN web service. It's a good test signal because:

- it's a real, well-documented earthquake with strong, unambiguous P and S arrivals (plus aftershocks), so both models should reliably produce picks
- it's freely available with no authentication
- it's the exact dataset SeisBench's maintainers use in their own ["applied picking" example notebook](https://github.com/seisbench/seisbench/blob/main/examples/02a_deploy_model_on_streams_example.ipynb), so this test doubles as a check against the upstream reference behavior

If the container has no outbound network access (or INGV happens to be unreachable), the data cell falls back to ObsPy's bundled example trace (`obspy.read()`, station BW.RJOB) so the test still runs offline -- it's a real but much smaller local earthquake recording, enough to confirm the models run and produce *some* output, just with fewer arrivals to pick.

To point this at a different event instead -- e.g. something recorded by an EarthScope/IRIS station -- swap the `Client("INGV")` call below for `Client("IRIS")` with a station/time of interest.

## Setup

In [ ]:
import obspy
import seisbench
import torch

print(f"seisbench {seisbench.__version__}")
print(f"torch {torch.__version__}")
print(f"obspy {obspy.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Load test waveform

In [ ]:
from obspy import UTCDateTime
from obspy.clients.fdsn import Client

try:
    client = Client("INGV")
    t = UTCDateTime(2009, 4, 6, 1, 30)
    stream = client.get_waveforms(
        network="MN",
        station="AQU",
        location="*",
        channel="HH?",
        starttime=t,
        endtime=t + 3600,
    )
    data_source = "INGV MN.AQU -- 2009 L'Aquila mainshock + aftershocks"
except Exception as exc:
    print(f"Live FDSN fetch failed ({type(exc).__name__}: {exc}); "
          f"falling back to ObsPy's bundled example trace.")
    stream = obspy.read()
    data_source = "ObsPy bundled example (BW.RJOB)"

print(f"Using: {data_source}")
print(stream)

## Load pretrained models

In [ ]:
import seisbench.models as sbm

pn_model = sbm.PhaseNet.from_pretrained("stead")
eqt_model = sbm.EQTransformer.from_pretrained("original")

# Moves each model to CUDA/MPS/XPU if available, else leaves it on CPU.
pn_model.to_preferred_device(verbose=True)
eqt_model.to_preferred_device(verbose=True)

## Run inference

`classify()` handles resampling to each model's expected rate internally, so the raw stream can be passed straight in.

In [ ]:
pn_outputs = pn_model.classify(stream)
eqt_outputs = eqt_model.classify(stream)

print(f"PhaseNet:      {len(pn_outputs.picks)} picks")
print(f"EQTransformer: {len(eqt_outputs.picks)} picks")

for name, outputs in [("PhaseNet", pn_outputs), ("EQTransformer", eqt_outputs)]:
    print(f"\n{name} -- first 5 picks:")
    for pick in outputs.picks[:5]:
        print(f"  {pick}")

## Pass/fail summary

In [ ]:
results = {"PhaseNet": len(pn_outputs.picks), "EQTransformer": len(eqt_outputs.picks)}

print(f"Data source: {data_source}")
print(f"Model device: {pn_model.device}")
print()
for name, n_picks in results.items():
    status = "PASS" if n_picks > 0 else "FAIL"
    print(f"[{status}] {name}: {n_picks} picks")

assert all(n > 0 for n in results.values()), (
    "At least one model produced zero picks -- seisbench likely isn't "
    "working correctly in this environment."
)
print("\nAll models produced picks -- seisbench is working.")

## Optional: visualize detection probabilities

Not required for the pass/fail check above -- useful when eyeballing the result interactively in JupyterLab.

In [ ]:
import matplotlib.pyplot as plt

pn_preds = pn_model.annotate(stream)

color = {"P": "C0", "S": "C1", "Detection": "C2"}
fig, ax = plt.subplots(figsize=(15, 4))
for tr in pn_preds:
    _, phase = tr.stats.channel.split("_")
    if phase == "N":
        continue
    ax.plot(tr.times(), tr.data, label=phase, c=color.get(phase, "C3"))
ax.set_ylim(0, 1.05)
ax.set_xlabel("Time [s]")
ax.set_ylabel("Probability")
ax.set_title(f"PhaseNet detections -- {data_source}")
ax.legend()
plt.show()